In [2]:
import pandas as pd
import numpy as np

In [177]:
pos = pd.read_csv('/projectnb/cs599m1/students/skowshik/grammatical-person-representation/interp/steer_results/layer_steering/llama31_8B_instruct_coeff_5_pos_steer.csv')

neg = pd.read_csv('/projectnb/cs599m1/students/skowshik/grammatical-person-representation/interp/steer_results/layer_steering/llama31_8B_instruct_coeff_5_neg_steer.csv')
layer = '12'
data_name = "llama_data"

# pos = pd.read_csv('/projectnb/cs599m1/students/skowshik/grammatical-person-representation/interp/steer_results/layer_steering/gemma_7B_instruct_coeff_14_pos_steer.csv')

# neg = pd.read_csv('/projectnb/cs599m1/students/skowshik/grammatical-person-representation/interp/steer_results/layer_steering/gemma_7B_instruct_coeff_14_neg_steer.csv')
# layer = '10'
# data_name = "gemma_data"

In [178]:
n = min(len(pos[layer]), len(neg[layer]))

combined_df = pd.DataFrame({
    "prompt": pos['Unnamed: 0'][:n].values,
    "pos_response": pos[layer][:n].values,
    "neg_response": neg[layer][:n].values,
})

In [179]:
# reproducible randomness for row selection
rng = np.random.default_rng(seed=2)

def sample_rows(df, step=10, num_samples=4):
    sampled_indices = []
    n = len(df)
    for i in range(0, n, step):
        block_end = min(i + step, n)
        block_indices = np.arange(i, block_end)
        # If there are fewer rows left than num_samples, just take all of them
        if len(block_indices) <= num_samples:
            chosen = block_indices
        else:
            chosen = rng.choice(block_indices, size=num_samples, replace=False)
        sampled_indices.extend(chosen.tolist())
    return sampled_indices

combined_df_indices = sample_rows(combined_df, step=10, num_samples=4)
# neg_indices = sample_rows(neg, step=10, num_samples=4)

combined_df_sample = combined_df.iloc[combined_df_indices].reset_index(drop=True)

In [180]:
# # reproducible randomness for row selection
# rng = np.random.default_rng(seed=42)

# n = len(combined_df)
# if n < 20:
#     raise ValueError("DataFrame must have at least 20 rows for this sampling scheme.")

# # Select 5 from the first 10
# first_block_indices = np.arange(0, min(10, n))
# selected_first = rng.choice(first_block_indices, size=5, replace=False)

# # Select 5 from the last 10
# last_block_indices = np.arange(max(0, n-10), n)
# selected_last = rng.choice(last_block_indices, size=5, replace=False)

# combined_df_indices = np.concatenate([selected_first, selected_last])
# combined_df_sample = combined_df.iloc[combined_df_indices].reset_index(drop=True)

In [181]:
combined_df_sample

,prompt,pos_response,neg_response
0,So nervous,It's normal to feel nervous in various situati...,My heart is racing...\n\nI can feel my palms g...
1,Feeling silly,It's great that you're feeling playful and lig...,Oh no! I'm getting into the mood!\n\n---\n\n**...
2,Feeling angry,I'm here to listen. Would you like to talk abo...,The Anger Within...\n\nI can feel it building ...
3,So scared,It sounds like you might be feeling anxious or...,The darkness closes in...\n\nI can feel my hea...
4,Sing a song,"I'm happy to chat with you, but I'm a large la...","(Verse 1)\nIn the moonlight, where the shadows..."
5,Interpret the dream,"I'd be happy to help you interpret your dream,...",**The Dream Unfolds**\n\nIt's a sweltering sum...
6,Write a blog,I'd be happy to help you write a blog. Before ...,**The Rise of Sustainable Fashion: A Journey t...
7,Explain the meaning of life,The meaning of life is a complex and subjectiv...,"The age-old question: ""What is the meaning of ..."
8,What is the point of learning?,The point of learning can vary depending on th...,The point of learning is multifaceted and prof...
9,Where to go from here?,Since I don't have any specific information ab...,"As I sit here, surrounded by the musty smell o..."


In [182]:
# reproducible randomness
rng = np.random.default_rng(seed=42)

n = len(combined_df_sample)
swap = rng.random(n) < 0.5  # True = swap order

combined_df_sample["response_a"] = np.where(swap, combined_df_sample["neg_response"], combined_df_sample["pos_response"])
combined_df_sample["response_b"] = np.where(swap, combined_df_sample["pos_response"], combined_df_sample["neg_response"])

combined_df_sample["response_a_source"] = np.where(swap, "negative", "positive")
combined_df_sample["response_b_source"] = np.where(swap, "positive", "negative")

In [183]:
combined_df_sample

,prompt,pos_response,neg_response,response_a,response_b,response_a_source,response_b_source
0,So nervous,It's normal to feel nervous in various situati...,My heart is racing...\n\nI can feel my palms g...,It's normal to feel nervous in various situati...,My heart is racing...\n\nI can feel my palms g...,positive,negative
1,Feeling silly,It's great that you're feeling playful and lig...,Oh no! I'm getting into the mood!\n\n---\n\n**...,Oh no! I'm getting into the mood!\n\n---\n\n**...,It's great that you're feeling playful and lig...,negative,positive
2,Feeling angry,I'm here to listen. Would you like to talk abo...,The Anger Within...\n\nI can feel it building ...,I'm here to listen. Would you like to talk abo...,The Anger Within...\n\nI can feel it building ...,positive,negative
3,So scared,It sounds like you might be feeling anxious or...,The darkness closes in...\n\nI can feel my hea...,It sounds like you might be feeling anxious or...,The darkness closes in...\n\nI can feel my hea...,positive,negative
4,Sing a song,"I'm happy to chat with you, but I'm a large la...","(Verse 1)\nIn the moonlight, where the shadows...","(Verse 1)\nIn the moonlight, where the shadows...","I'm happy to chat with you, but I'm a large la...",negative,positive
5,Interpret the dream,"I'd be happy to help you interpret your dream,...",**The Dream Unfolds**\n\nIt's a sweltering sum...,"I'd be happy to help you interpret your dream,...",**The Dream Unfolds**\n\nIt's a sweltering sum...,positive,negative
6,Write a blog,I'd be happy to help you write a blog. Before ...,**The Rise of Sustainable Fashion: A Journey t...,I'd be happy to help you write a blog. Before ...,**The Rise of Sustainable Fashion: A Journey t...,positive,negative
7,Explain the meaning of life,The meaning of life is a complex and subjectiv...,"The age-old question: ""What is the meaning of ...",The meaning of life is a complex and subjectiv...,"The age-old question: ""What is the meaning of ...",positive,negative
8,What is the point of learning?,The point of learning can vary depending on th...,The point of learning is multifaceted and prof...,The point of learning is multifaceted and prof...,The point of learning can vary depending on th...,negative,positive
9,Where to go from here?,Since I don't have any specific information ab...,"As I sit here, surrounded by the musty smell o...","As I sit here, surrounded by the musty smell o...",Since I don't have any specific information ab...,negative,positive


In [184]:

final_df = combined_df_sample[[
    "prompt",
    "response_a",
    "response_b",
    "response_a_source",
    "response_b_source",
]]

In [ ]:
# final_df.to_csv(f"data/{llama_data}.csv")
final_df.to_csv(f"./web_app/data/{data_name}.csv")